In [104]:
import pandas as pd
pd.set_option("display.max_columns",50)

In [105]:
df = pd.read_csv(r"C:\Users\User\OneDrive\Desktop\DA Projects\job-market-dashboard\data\raw\postings.csv")

In [106]:
print(df.shape)

(123849, 31)


In [107]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  object 
 2   title                       123849 non-null  object 
 3   description                 123842 non-null  object 
 4   max_salary                  29793 non-null   float64
 5   pay_period                  36073 non-null   object 
 6   location                    123849 non-null  object 
 7   company_id                  122132 non-null  float64
 8   views                       122160 non-null  float64
 9   med_salary                  6280 non-null    float64
 10  min_salary                  29793 non-null   float64
 11  formatted_work_type         123849 non-null  object 
 12  applies                     23320 non-null   float64
 13  original_liste

In [108]:
# transpose for better readability
print(df.head(1).T)

                                                                            0
job_id                                                                 921716
company_name                                            Corcoran Sawyer Smith
title                                                   Marketing Coordinator
description                 Job descriptionA leading real estate firm in N...
max_salary                                                               20.0
pay_period                                                             HOURLY
location                                                        Princeton, NJ
company_id                                                          2774458.0
views                                                                    20.0
med_salary                                                                NaN
min_salary                                                               17.0
formatted_work_type                                             

In [109]:
# Missing valuse as percentage
print((df.isna().mean() * 100).round(1).sort_values(ascending = False))

closed_time                   99.1
skills_desc                   98.0
med_salary                    94.9
remote_allowed                87.7
applies                       81.2
max_salary                    75.9
min_salary                    75.9
currency                      70.9
compensation_type             70.9
pay_period                    70.9
normalized_salary             70.9
posting_domain                32.3
application_url               29.6
formatted_experience_level    23.7
fips                          22.1
zip_code                      16.9
views                          1.4
company_id                     1.4
company_name                   1.4
listed_time                    0.0
work_type                      0.0
sponsored                      0.0
job_id                         0.0
expiry                         0.0
application_type               0.0
original_listed_time           0.0
formatted_work_type            0.0
location                       0.0
description         

In [110]:
# Duplicates
print("Exact duplicates:", df.duplicated().sum())

Exact duplicates: 0


In [111]:
# Cardinality: how many unique values per column
print(df.nunique().sort_values())

remote_allowed                     1
sponsored                          1
compensation_type                  1
application_type                   4
pay_period                         5
currency                           6
formatted_experience_level         6
work_type                          7
formatted_work_type                7
applies                          274
views                            684
closed_time                      698
med_salary                      1417
skills_desc                     2212
fips                            2270
posting_domain                  4443
min_salary                      4612
max_salary                      5321
zip_code                        6989
normalized_salary               7259
location                        8526
company_name                   24428
company_id                     24474
listed_time                    53231
expiry                         54851
original_listed_time           65036
title                          72521
a


| Issue | Column(s) | Evidence | Fix (phase) |
| :--- | :--- | :--- | :--- |
| Covers all job types | `title` | 123,849 rows, all industries | Filter to data and analytics titles (Phase 4, A.4) |
| Salary mostly missing, split across two forms | `min_salary`, `max_salary`, `med_salary` | 75.9% missing min/max, 94.9% missing med | `salary_mid` = median if present, else mean of min and max; leave blanks as NULL |
| Mixed pay periods and currencies | `pay_period`, `currency` | 5 pay periods, 6 currency values | Annualize, keep USD only |
| Experience level missing | `formatted_experience_level` | 23.7% missing | Fill with "Not specified" |
| Remote flag has one value | `remote_allowed` | 87.7% missing, only value is 1 | Fill blanks with 0 |
| Dates are epoch milliseconds | `original_listed_time` | float, no missing | Convert with `unit="ms"` |
| Location is "City, ST" or just "United States" | `location` | 8,526 unique values | Split into city and state; city blank when no state |
| Repeated descriptions | `description` | 107,827 unique of 123,842 | Check duplicates within your filtered data, then decide and document |
| Missing company and 7 missing descriptions | `company_name`, `description` | 1.4% and 7 rows | Keep rows, blank company; no skills for empty descriptions |
| Very large job IDs | `job_id` | up to about 3.9 billion | `BIGINT` in SQL Server, keep LinkedIn's ID |
| Unneeded columns | `urls`, `expiry`, `closed_time`, `zip_code`, `fips`, `posting_domain`, `sponsored`, `compensation_type` | no analytic use | Drop in Phase 4 |


In [112]:
df.sample(20)

,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,min_salary,formatted_work_type,applies,original_listed_time,remote_allowed,job_posting_url,application_url,application_type,expiry,closed_time,formatted_experience_level,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
82703,3903897936,Spectrum Enterprise,"Strategic Account Specialist - Healthcare, Spe...",Do you want to focus on daily prospecting to g...,NaN,NaN,"Columbus, OH",2032.0,4.0,NaN,NaN,Full-time,NaN,1.713398e+12,NaN,https://www.linkedin.com/jobs/view/3903897936/...,https://jobs.spectrum.com/job/-/-/4673/6411821...,OffsiteApply,1.716094e+12,NaN,Entry level,NaN,1.713502e+12,jobs.spectrum.com,0,FULL_TIME,NaN,NaN,NaN,43085.0,39049.0
22689,3889727474,A Hiring Company,Middle School Spanish and French Teacher,\n\nABOUT SEWICKLEY ACADEMY\nDistinguished by ...,NaN,NaN,"Sewickley, PA",101478385.0,7.0,NaN,NaN,Full-time,NaN,1.712665e+12,NaN,https://www.linkedin.com/jobs/view/3889727474/...,https://www.click2apply.net/Y24yoyFPbx5dYCpgBT...,OffsiteApply,1.715257e+12,NaN,NaN,\nPI239308501,1.712665e+12,www.click2apply.net,0,FULL_TIME,NaN,NaN,NaN,15143.0,42003.0
112352,3905647699,U.S. Department of Veterans Affairs,Cemetery Admin Specialist,Summary\n\nThis position is located at the Dal...,NaN,NaN,"Dallas, TX",23789.0,2.0,NaN,NaN,Full-time,NaN,1.713513e+12,NaN,https://www.linkedin.com/jobs/view/3905647699/...,https://www.usajobs.gov:443/job/787292400,OffsiteApply,1.716105e+12,NaN,Entry level,NaN,1.713513e+12,www.usajobs.gov,0,FULL_TIME,NaN,NaN,NaN,75201.0,48113.0
36538,3895570177,Cracker Barrel,Restaurant Manager,"Overview:\n\nAt Cracker Barrel, you'll be join...",51000.0,YEARLY,"Corpus Christi, TX",12534.0,5.0,NaN,13000.0,Full-time,NaN,1.712894e+12,NaN,https://www.linkedin.com/jobs/view/3895570177/...,https://click.appcast.io/track/j5lrr84-org?cs=...,OffsiteApply,1.715487e+12,NaN,NaN,NaN,1.712895e+12,click.appcast.io,0,FULL_TIME,USD,BASE_SALARY,32000.0,78401.0,48355.0
28602,3894210135,USAA,Marketing Manager Lead - Bank Affiliate Market...,Why USAA?\n\nLet’s do something that really ma...,NaN,NaN,"San Antonio, TX",4501.0,8.0,NaN,NaN,Full-time,1.0,1.712855e+12,1.0,https://www.linkedin.com/jobs/view/3894210135/...,https://dsp.prng.co/KnDzWVb,OffsiteApply,1.715448e+12,NaN,Mid-Senior level,NaN,1.712856e+12,dsp.prng.co,0,FULL_TIME,NaN,NaN,NaN,78201.0,48029.0
96258,3904944579,GE Vernova,"Wind Hub Technician- Snyder, TX",Job Description Summary\n\nGE’s Onshore wind p...,NaN,NaN,"Texas, United States",82315716.0,4.0,NaN,NaN,Full-time,NaN,1.713453e+12,1.0,https://www.linkedin.com/jobs/view/3904944579/...,https://jobs.gecareers.com/vernova/global/en/j...,OffsiteApply,1.716045e+12,NaN,Associate,NaN,1.713453e+12,jobs.gecareers.com,0,FULL_TIME,NaN,NaN,NaN,NaN,NaN
101178,3905243108,The Rockefeller University,Staff Accountant,The Rockefeller University is the world's lead...,75000.0,YEARLY,New York City Metropolitan Area,8868.0,4.0,NaN,65000.0,Full-time,1.0,1.713464e+12,NaN,https://www.linkedin.com/jobs/view/3905243108/...,https://careers-rockefelleruniversity.icims.co...,OffsiteApply,1.716056e+12,NaN,NaN,NaN,1.713464e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,70000.0,NaN,NaN
38996,3898162783,Cardinal Group Companies,Community Manager,"POSITION: Community Manager (Full-Time, Exempt...",NaN,NaN,"Austin, TX",2214607.0,12.0,NaN,NaN,Full-time,2.0,1.713139e+12,NaN,https://www.linkedin.com/jobs/view/3898162783/...,https://recruiting2.ultipro.com/CAR1054CGMA/Jo...,OffsiteApply,1.715803e+12,NaN,Mid-Senior level,NaN,1.713211e+12,recruiting2.ultipro.com,0,FULL_TIME,NaN,NaN,NaN,78701.0,48453.0
54521,3901680099,Ascendo Resources,Office Administrator,Exciting Opportunity!\nDive into an amazing op...,125000.0,YEARLY,"Tampa, FL",777243.0,15.0,NaN,115000.0,Full-time,6.0,1.713213e+12,NaN,https://www.linkedin.com/jobs/view/3901680099/...,NaN,ComplexOnsiteApply,1.715805e+12,NaN,Mid-Senior level,NaN,1.713213e+

In [113]:
# 1.Salary logic
print("Rows with median and min/max",(df["med_salary"].notna() & df["min_salary"].notna() & df["max_salary"].notna()).sum())
print("pay period is filled but no salary",(df["pay_period"].notna() & df[["med_salary","min_salary","max_salary"]].isna().all(axis=1)).sum())
print(df.groupby("pay_period")[["med_salary","min_salary","max_salary"]].median())

Rows with median and min/max 0
pay period is filled but no salary 0
            med_salary  min_salary  max_salary
pay_period                                    
BIWEEKLY           NaN     54009.0     66976.0
HOURLY            21.0        25.0        32.0
MONTHLY         2364.5      5550.0      7350.0
WEEKLY             NaN      2099.0      2099.0
YEARLY         71905.0     87000.0    120000.0


In [114]:
# 2.Dates
d = pd.to_datetime(df["original_listed_time"],unit = "ms")
print(d.min(),d.max())
print(d.dt.to_period("M").value_counts().sort_index())

2023-12-05 21:08:53 2024-04-20 00:26:43
original_listed_time
2023-12         3
2024-01         7
2024-02         9
2024-03      1815
2024-04    122015
Freq: M, Name: count, dtype: int64


In [115]:
# 3.Category values
for c in ["formatted_experience_level","formatted_work_type","currency"]:
    print(df[c].value_counts(dropna=False),"\n")

formatted_experience_level
Mid-Senior level    41489
Entry level         36708
NaN                 29409
Associate            9826
Director             3746
Internship           1449
Executive            1222
Name: count, dtype: int64 

formatted_work_type
Full-time     98814
Contract      12117
Part-time      9696
Temporary      1190
Internship      983
Volunteer       562
Other           487
Name: count, dtype: int64 

currency
NaN    87776
USD    36058
EUR        6
CAD        3
BBD        2
AUD        2
GBP        2
Name: count, dtype: int64 



In [116]:
# 4. Location shape and look at real rows
print(df["location"].str.match(r"^[^,]+,\s*[A-Z]{2}$").mean())
df[["title","company_name","location","pay_period","min_salary","max_salary",
    "formatted_experience_level"]].sample(20, random_state=1)

0.8485332945764601


,title,company_name,location,pay_period,min_salary,max_salary,formatted_experience_level
109896,Trademark Attorney,Marina Sirras & Associates LLC,"New York, NY",YEARLY,225000.0,260000.0,Associate
15087,Delivery Driver / CDL A required / Seasonal,PepsiCo,"Alexandria, LA",HOURLY,NaN,NaN,NaN
101176,Senior Recruiter,The Armstrong Company,Memphis Metropolitan Area,NaN,NaN,NaN,NaN
106314,Order Fulfillment Coordinator,Allied OneSource,"Jacksonville, FL",NaN,NaN,NaN,Entry level
79678,Medical Assistant Urgent Care Per Diem,Carbon Health,"Beverly, MA",HOURLY,20.0,25.0,Entry level
91020,Endoscopy Technician,RUSH University Medical Center,"Chicago, IL",NaN,NaN,NaN,Entry level
53258,OT and DCS Team Lead,bp,"Chicago, IL",NaN,NaN,NaN,Mid-Senior level
107336,Production Supervisor - 2nd Shift,Tyson Foods,"Nashville, AR",NaN,NaN,NaN,NaN
48319,Associate Software Developer (Java/Scala),FlexTrade,"Milwaukee, WI",NaN,NaN,NaN,Associate
69185,Senior Information Technology Project Manager,Selby Jennings,"Boston, MA",NaN,NaN,NaN,Mid-Senior level


# Cleaning

## Load only needed Columns

In [117]:

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
RAW = "../data/raw/"

use = ["job_id", "company_name", "title", "description", "location",
       "min_salary", "med_salary", "max_salary", "pay_period", "currency",
       "formatted_experience_level", "work_type", "remote_allowed",
       "original_listed_time"]
df = pd.read_csv(RAW + "postings.csv", usecols=use)
print(df.shape)

(123849, 14)


## scope the data to data role

In [118]:
include = (r"data analyst|business analyst|\bBI\b|analytics|reporting analyst|insights analyst|"
           r"data scientist|data engineer|machine learning|Power BI|Tableau|\bSQL\b|"
           r"data scien|insights analy|data analy")
exclude = (r"data entry|behavior|financial analyst|credit analyst|tax analyst|compliance|"
           r"online data|database admin|financial|\bdba\b|full stack|"
           r"lingual|\bBI/UM\b|attorney|legal")   # last line: Bi-Lingual and legal BI/UM false matches

is_data  = df["title"].str.contains(include, case=False, regex=True, na=False)
not_data = df["title"].str.contains(exclude, case=False, regex=True, na=False)

n_all = len(df)
scoped = df[is_data & ~not_data].copy()
n_title = len(scoped)
scoped = scoped.drop_duplicates(
    subset=["title", "company_name", "location", "original_listed_time"]).copy()
n_dedup = len(scoped)
print(f"All rows: {n_all:,} | after title filter: {n_title:,} | after de-duplication: {n_dedup:,}")

All rows: 123,849 | after title filter: 2,331 | after de-duplication: 2,318


## Rename Columns

In [119]:
scoped = scoped.rename(columns={
    "company_name": "company", "title": "job_title",
    "max_salary": "salary_max", "med_salary": "salary_med", "min_salary": "salary_min",
    "original_listed_time": "listed_time", "formatted_experience_level": "exp_level"})
print(scoped["job_id"].is_unique)

True


## Dates

In [120]:
scoped["posted_date"] = pd.to_datetime(scoped["listed_time"], unit="ms").dt.normalize()
print(scoped["posted_date"].min(), scoped["posted_date"].max(), scoped["posted_date"].isna().sum())
print(scoped["posted_date"].dt.to_period("M").value_counts().sort_index())

2024-01-08 00:00:00 2024-04-20 00:00:00 0
posted_date
2024-01       1
2024-03      36
2024-04    2281
Freq: M, Name: count, dtype: int64


## Text and Location

In [121]:
STATE_CODES = {
 "Alabama":"AL","Alaska":"AK","Arizona":"AZ","Arkansas":"AR","California":"CA","Colorado":"CO",
 "Connecticut":"CT","Delaware":"DE","District of Columbia":"DC","Florida":"FL","Georgia":"GA",
 "Hawaii":"HI","Idaho":"ID","Illinois":"IL","Indiana":"IN","Iowa":"IA","Kansas":"KS",
 "Kentucky":"KY","Louisiana":"LA","Maine":"ME","Maryland":"MD","Massachusetts":"MA",
 "Michigan":"MI","Minnesota":"MN","Mississippi":"MS","Missouri":"MO","Montana":"MT",
 "Nebraska":"NE","Nevada":"NV","New Hampshire":"NH","New Jersey":"NJ","New Mexico":"NM",
 "New York":"NY","North Carolina":"NC","North Dakota":"ND","Ohio":"OH","Oklahoma":"OK",
 "Oregon":"OR","Pennsylvania":"PA","Rhode Island":"RI","South Carolina":"SC",
 "South Dakota":"SD","Tennessee":"TN","Texas":"TX","Utah":"UT","Vermont":"VT",
 "Virginia":"VA","Washington":"WA","West Virginia":"WV","Wisconsin":"WI","Wyoming":"WY"}

for c in ["job_title", "company", "location"]:
    scoped[c] = scoped[c].str.strip()

def parse_location(loc):
    """Return (city, state_code). Handles the four formats in this dataset."""
    parts = [p.strip() for p in str(loc).split(",")]
    if len(parts) == 1:                          # "United States", "Atlanta Metropolitan Area"
        return pd.NA, pd.NA
    city, second = parts[0], parts[1]
    if second == "United States":                # "Michigan, United States": a state, not a city
        return pd.NA, STATE_CODES.get(city, pd.NA)
    second = second.replace(" Metropolitan Area", "")   # "Austin, Texas Metropolitan Area"
    if len(second) == 2 and second.isupper():    # "Cleveland, OH"
        return city, second
    state = STATE_CODES.get(second, pd.NA)       # "Los Angeles, California, United States"
    return (city if pd.notna(state) else pd.NA), state

parsed = scoped["location"].apply(parse_location)
scoped["city"] = parsed.str[0]
scoped["state"] = parsed.str[1]

unparsed = scoped.loc[scoped["state"].isna(), "location"]
print("Unparsed locations:", len(unparsed))
print(unparsed.value_counts().head(8))
print("Missing state:", scoped["state"].isna().sum(), "| missing city:", scoped["city"].isna().sum())
print("States longer than 2 chars:", scoped.loc[scoped["state"].str.len() > 2, "state"].unique())
print(scoped["state"].value_counts().head(15))
print(scoped["city"].value_counts().head(15))

Unparsed locations: 524
location
United States                      388
New York City Metropolitan Area     31
San Francisco Bay Area              12
Washington DC-Baltimore Area        12
Atlanta Metropolitan Area            9
Greater Chicago Area                 7
Dallas-Fort Worth Metroplex          6
Los Angeles Metropolitan Area        5
Name: count, dtype: int64
Missing state: 524 | missing city: 648
States longer than 2 chars: []
state
CA    222
TX    209
NY    130
VA    129
NJ    106
FL    101
IL     89
NC     78
WA     70
GA     68
MA     65
PA     59
OH     48
MI     36
AZ     35
Name: count, dtype: int64
city
New York         88
Atlanta          53
Chicago          51
Austin           39
San Francisco    37
Boston           36
Dallas           36
Charlotte        36
Seattle          32
Houston          31
McLean           30
Tampa            30
Los Angeles      27
Richmond         27
Plano            27
Name: count, dtype: int64


## Role Family

In [122]:
import re

def role_family(title):
    t = str(title).lower()
    if "data scien" in t or "machine learning" in t:
        return "Data Scientist / ML"
    if "data engineer" in t or "analytics engineer" in t or "etl" in t:
        return "Data Engineer"
    if ("business analy" in t or "business intelligence" in t or "power bi" in t
            or "powerbi" in t or "tableau" in t or "reporting analy" in t
            or re.search(r"\bbi\b", t) or "sql" in t):
        return "Business / BI Analyst"
    if "data analy" in t or "analytics" in t or "insights" in t:
        return "Data Analyst"
    return "Other"

scoped["role_family"] = scoped["job_title"].apply(role_family)
print(scoped["role_family"].value_counts())
print(scoped.loc[scoped["role_family"] == "Other", "job_title"].head(15).tolist())

role_family
Business / BI Analyst    764
Data Analyst             648
Data Scientist / ML      483
Data Engineer            423
Name: count, dtype: int64
[]


## Salary - Annual USD
1. Convert every pay period to annual. 2. One mid-point. 3. USD only. 4. Trim outliers (1st to 99th percentile).

In [123]:
periods = {"HOURLY": 2080, "WEEKLY": 52, "BIWEEKLY": 26, "MONTHLY": 12, "YEARLY": 1}
factor = scoped["pay_period"].str.upper().map(periods)
for c in ["salary_min", "salary_max", "salary_med"]:
    scoped[c] = scoped[c] * factor

scoped["salary_mid"] = scoped["salary_med"].fillna(scoped[["salary_min", "salary_max"]].mean(axis=1))

# Non-USD pay is dropped (job stays). Rows with no currency also have no salary.
non_usd = scoped["currency"].notna() & (scoped["currency"] != "USD")
print("Non-USD salary rows:", non_usd.sum())
scoped.loc[non_usd, ["salary_min", "salary_max", "salary_med", "salary_mid"]] = np.nan

lo, hi = scoped["salary_mid"].quantile([0.01, 0.99])
outliers = ~scoped["salary_mid"].between(lo, hi) & scoped["salary_mid"].notna()
print(f"Trimming outside {lo:,.0f} to {hi:,.0f}: {outliers.sum()} rows")
scoped.loc[outliers, ["salary_min", "salary_max", "salary_med", "salary_mid"]] = np.nan

print("Salary coverage:", round(scoped["salary_mid"].notna().mean() * 100, 1), "%")
print(scoped["salary_mid"].describe().round(0))

Non-USD salary rows: 0
Trimming outside 9,420 to 300,000: 13 rows
Salary coverage: 31.1 %
count       720.0
mean     134461.0
std       48220.0
min       29120.0
25%       99962.0
50%      127525.0
75%      163638.0
max      300000.0
Name: salary_mid, dtype: float64


Optional cross-check against LinkedIn's own annual figure (`normalized_salary`):

In [124]:
chk = pd.read_csv(RAW + "postings.csv", usecols=["job_id", "normalized_salary"])
chk = scoped[["job_id", "salary_mid"]].merge(chk, on="job_id").dropna()
chk["diff_pct"] = (chk["salary_mid"] - chk["normalized_salary"]).abs() / chk["normalized_salary"] * 100
print(len(chk), "rows compared | median difference %:", round(chk["diff_pct"].median(), 2))

720 rows compared | median difference %: 0.0


## Experience level and remote flag

In [125]:
scoped["exp_level"] = scoped["exp_level"].fillna("Not specified")
order = {"Internship": 1, "Entry level": 2, "Associate": 3, "Mid-Senior level": 4,
         "Director": 5, "Executive": 6, "Not specified": 7}
scoped["exp_level_order"] = scoped["exp_level"].map(order)

scoped["remote_allowed"] = scoped["remote_allowed"].fillna(0).astype(int)
print(scoped["exp_level"].value_counts())
print(scoped["remote_allowed"].value_counts())
print(scoped["work_type"].value_counts())

exp_level
Mid-Senior level    1076
Not specified        679
Entry level          288
Associate            161
Director              67
Internship            34
Executive             13
Name: count, dtype: int64
remote_allowed
0    1650
1     668
Name: count, dtype: int64
work_type
FULL_TIME     1608
CONTRACT       615
INTERNSHIP      54
PART_TIME       20
TEMPORARY       10
OTHER            8
VOLUNTEER        3
Name: count, dtype: int64


## Industry
A job can have several industries; we keep the first one listed (a simplification to state in the README).

In [126]:
ji = pd.read_csv(RAW + "jobs/job_industries.csv")
im = pd.read_csv(RAW + "mappings/industries.csv")
ind = (ji.merge(im, on="industry_id")
         .drop_duplicates("job_id")
         .rename(columns={"industry_name": "industry"})[["job_id", "industry"]])
scoped = scoped.merge(ind, on="job_id", how="left")
print("Missing industry:", scoped["industry"].isna().sum())
print(scoped["industry"].value_counts().head(10))

Missing industry: 16
industry
IT Services and IT Consulting        519
Financial Services                   250
Software Development                 232
Hospitals and Health Care            106
Staffing and Recruiting              106
Information Services                  66
Insurance                             65
Banking                               60
Information Technology & Services     52
Technology, Information and Media     50
Name: count, dtype: int64


## Final Checks and Save

In [127]:
cols = ["job_id", "job_title", "company", "location", "city", "state", "industry",
        "role_family", "exp_level", "exp_level_order", "work_type", "remote_allowed",
        "salary_min", "salary_max", "salary_mid", "posted_date", "description"]
scoped = scoped[cols]

print(scoped.shape)
print(scoped.dtypes)
print((scoped.isna().mean() * 100).round(1))
print("job_id unique:", scoped["job_id"].is_unique)

scoped.to_csv("../data/clean/jobs_clean.csv", index=False)

(2318, 17)
job_id                      int64
job_title                  object
company                    object
location                   object
city                       object
state                      object
industry                   object
role_family                object
exp_level                  object
exp_level_order             int64
work_type                  object
remote_allowed              int64
salary_min                float64
salary_max                float64
salary_mid                float64
posted_date        datetime64[ns]
description                object
dtype: object
job_id              0.0
job_title           0.0
company             1.0
location            0.0
city               28.0
state              22.6
industry            0.7
role_family         0.0
exp_level           0.0
exp_level_order     0.0
work_type           0.0
remote_allowed      0.0
salary_min         71.2
salary_max         71.2
salary_mid         68.9
posted_date         0.0
description  